Trains a VAE celeb-a geneartor

In [ ]:
# Colab Setup: clone repo and install requirements
import os, sys

# check if running in Colab
IN_COLAB = "google.colab" in sys.modules
print(f"Running on Colab:{IN_COLAB}")

if IN_COLAB:
    if not os.path.exists("vis10n"):
        print("Cloning full repo")
        !git clone https://github.com/simchaorgel/vis10n.git
    !cd vis10n && git pull
    sys.path.append("vis10n")
    # Download requirements
    print("Downloading requirements")
    !pip install -q -r vis10n/requirements.txt
else:
    # Running locally from training/: make the repo root importable
    sys.path.append("..")

In [ ]:
# Main imports
import torch as t
from dataclasses import dataclass
from tqdm.notebook import tqdm
import wandb
from dataclasses import dataclass
from typing import Literal
from torch.utils.data import DataLoader, Subset
from jaxtyping import Float
from torch import Tensor, nn

In [ ]:
# Local imports: this cell imports the necessary files
# either locally or from colabs repo clone
from dataloaders import celeba
from models.mnist_vae import VAE

In [ ]:
# Check if we have access to a GPU
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Wandb login
wandb.login()

In [ ]:
@dataclass
class VAEArgs:
    # architecture
    latent_dim_size: int = 5
    hidden_dim_size: int = 128

    # data / training
    n_data_samples: int | None = 30_000
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    betas: tuple[float, float] = (0.5, 0.999)
    beta_kl: float = 0.1

    # logging
    use_wandb: bool = True
    wandb_project: str | None = "day5-autoencoder"
    wandb_name: str | None = None
    log_every_n_steps: int = 250

In [ ]:
class VAETrainer:
    def __init__(self, args: VAEArgs):
        self.args = args
        self.trainset = celeba.get_dataset(args.n_data_samples)
        self.trainloader = DataLoader(self.trainset, batch_size=args.batch_size, shuffle=True, num_workers=8)
        self.model = VAE(
            latent_dim_size=args.latent_dim_size,
            hidden_dim_size=args.hidden_dim_size,
        ).to(device)
        self.optimizer = t.optim.Adam(self.model.parameters(), lr=args.lr, betas=args.betas)

    def training_step(
        self, img: Float[Tensor, "batch 1 height width"]
    ) -> Float[Tensor, ""]:
        """
        Performs a training step on the batch of images in `img`. Returns the loss. Logs to wandb
        if enabled.
        """
        # Forward pass (1 batch)
        decoded_img, mu, logsigma = self.model(img)

        # Loss: difference between img | output, AND KL divergence
        kl_divergence = ((t.exp(logsigma)**2 + mu**2 - 1) / 2 - logsigma).mean() * self.args.beta_kl
        reconstruct_to_img_loss = nn.MSELoss()(decoded_img, img)
        loss =  reconstruct_to_img_loss + kl_divergence

        # Back pass
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        # Logging (loss) (increment step)
        self.step += 1

        if args.use_wandb:
            wandb.log(
                dict(
                    reconstruction_loss=reconstruct_to_img_loss.item(),
                    kl_div_loss=kl_divergence.item(),
                    mean=mu.mean(),
                    std=t.exp(logsigma).mean(),
                    total_loss=loss.item(),
                ),
                step=self.step)

        return loss

    @t.inference_mode()
    def log_samples(self) -> None:
        """
        Evaluates model on holdout data, either logging to wandb or displaying output inline.
        """
        assert self.step > 0, "First call should come after a training step. Remember to increment `self.step`."
        output = self.model(HOLDOUT_DATA)[0]
        if self.args.use_wandb:
            output = (output - output.min()) / (output.max() - output.min())  # Normalize to [0, 1]
            output = (output * 255).to(dtype=t.uint8)  # Convert to uint8 for logging
            wandb.log({"images": [wandb.Image(arr) for arr in output.cpu().numpy()]}, step=self.step)
        else:
            display_data(t.concat([HOLDOUT_DATA, output]), nrows=2, title="VAE reconstructions")

    def train(self) -> VAE:
        """Performs a full training run."""
        self.step = 0
        if self.args.use_wandb:
            wandb.init(project=self.args.wandb_project, name=self.args.wandb_name)
            wandb.watch(self.model)

         # Loop over epochs
        for epoch in range(self.args.epochs):
            progress_bar = tqdm(self.trainloader, ascii=True)
            # Loop over batches
            for images, labels in progress_bar:
                # Train step
                images = images.to(device)
                loss = self.training_step(images)
                progress_bar.set_description(f"{epoch=:02d}, {loss=:.4f}, step={self.step:05d}")

                # Every n steps, evaluate as well
                if self.step % self.args.log_every_n_steps == 0:
                    self.log_samples()

        if self.args.use_wandb:
            wandb.finish()

        return self.model

In [ ]:
args = VAEArgs(latent_dim_size=5, hidden_dim_size=100, use_wandb=True)
trainer = VAETrainer(args)
vae = trainer.train()

In [ ]:
grid_latent = create_grid_of_latents(vae, interpolation_range=(-1, 1))
output = vae.decoder(grid_latent)
utils.visualise_output(output, grid_latent, title="VAE latent space visualization")

In [ ]:
small_dataset = Subset(get_dataset("MNIST"), indices=range(0, 5000))
imgs = t.stack([img for img, label in small_dataset]).to(device)
labels = t.tensor([label for img, label in small_dataset]).to(device).int()

# We're getting the mean vector, which is the [0]-indexed output of the encoder
latent_vectors = vae.encoder(imgs)[0, :, :2]
holdout_latent_vectors = vae.encoder(HOLDOUT_DATA)[0, :, :2]

utils.visualise_input(latent_vectors, labels, holdout_latent_vectors, HOLDOUT_DATA)